In [44]:
import pandas as pd
import matplotlib.pyplot as plt

In [45]:
import os
from pathlib import Path

In [46]:
import xgboost as xgb

In [47]:
import seaborn as sns
import matplotlib.pyplot as plt

In [48]:
from sklearn.model_selection import GridSearchCV, cross_validate, validation_curve,train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score,accuracy_score, mean_absolute_error
import warnings
warnings.simplefilter(action='ignore', category=Warning)

In [49]:
DATA_PATH = Path("/Users/memeh/CodeProjects/IFT3710/kaggle/geolifeclef-2021/data")

In [50]:
df_obs_fr = pd.read_csv(DATA_PATH / "observations" / "observations_fr_train.csv", sep=";", index_col="observation_id")
df_obs_us = pd.read_csv(DATA_PATH / "observations" / "observations_us_train.csv", sep=";", index_col="observation_id")
df_obs = pd.concat((df_obs_fr, df_obs_us))

In [51]:
df_obs.head()

,latitude,longitude,species_id,subset
observation_id,,,,
10561949,45.705116,1.424622,241,train
10131188,45.146973,6.416794,101,train
10076047,49.746944,4.686389,38,train
10799362,46.783695,-2.072855,701,train
10392536,48.604866,-2.825003,1460,train


In [52]:
obs_id_train = df_obs.index[df_obs["subset"] == "train"].values
obs_id_val = df_obs.index[df_obs["subset"] == "val"].values

y_train = df_obs.loc[obs_id_train]["species_id"].values
y_val = df_obs.loc[obs_id_val]["species_id"].values

n_val = len(obs_id_val)
print("Validation set size: {} ({:.1%} of train observations)".format(n_val, n_val / len(df_obs)))

Validation set size: 45446 (2.4% of train observations)


In [53]:
df_obs_fr_test = pd.read_csv(DATA_PATH / "observations" / "observations_fr_test.csv", sep=";", index_col="observation_id")
df_obs_us_test = pd.read_csv(DATA_PATH / "observations" / "observations_us_test.csv", sep=";", index_col="observation_id")
df_obs_test = pd.concat((df_obs_fr_test, df_obs_us_test))

In [54]:
obs_id_test = df_obs_test.index.values
print("Number of observations for testing: {}".format(len(df_obs_test)))

df_obs_test.head()

Number of observations for testing: 42405


,latitude,longitude
observation_id,,
10782781,43.601788,6.940195
10364138,46.241711,0.683586
10692017,45.181095,1.533459
10222322,46.938450,5.298678
10241950,45.017433,0.960736


In [ ]:
from xgboost import XGBClassifier

In [56]:
x_train = df_obs.loc[obs_id_train].drop(columns=["species_id", "subset"])
x_val = df_obs.loc[obs_id_val].drop(columns=["species_id", "subset"])

In [ ]:
print("y_train unique species IDs range:", y_train.min(), "to", y_train.max())
print("y_val unique species IDs range:", y_val.min(), "to", y_val.max())
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Fit on ALL species ids so both train and val share the same mapping
le.fit(df_obs["species_id"].values)

y_train = le.transform(df_obs.loc[obs_id_train]["species_id"].values)
y_val   = le.transform(df_obs.loc[obs_id_val]["species_id"].values)
xgb_cls = xgb.XGBClassifier(objective="multi:softmax", eval_metric="mlogloss", n_estimators=100, early_stopping_rounds=5)
xgb_cls.fit(x_train, y_train, eval_set=[(x_train, y_train), (x_val, y_val)], verbose=0)

